# Scraping A-Z Animals (https://a-z-animals.com/animals/)

## Scraping para obtener el link de cada animal disponible en la página web

In [1]:
import cloudscraper
import time

# animals = [
#     "Tiger", "Lion", "Elephant", "Dolphin", "Giant Panda Bear", "Horse", "Penguin", "Wolf", "Shark", "Rabbit",
#     "Gorilla", "Giraffe", "Cheetah", "Polar Bear", "Hippopotamus", "Zebra", "Red Panda", "Kangaroo", "Koala", "Sloth",
#     "Meerkat", "Rhinoceros", "Snow Leopard", "Orangutan", "Chimpanzee", "Platypus", "Lemur", "Capybara", "Sea Otter", "Arctic Fox",
#     "Jaguar", "Panther", "Brown Bear", "Okapi", "Tasmanian Devil", "Wombat", "Quokka", "Hedgehog", "Fennec Fox", "Bald Eagle",
#     "Peacock", "Flamingo", "Parrot", "Hummingbird", "Swan", "Toucan", "Owl", "Puffin", "Woodpecker", "Common Raven",
#     "Falcon", "Ostrich", "Emu", "Albatross", "Robin", "Blue Jay", "Cardinal", "Squirrel", "Guinea Pig", "Hamster",
#     "Cow", "Pig", "Sheep", "Goat", "Chicken", "Duck", "Donkey", "Llama", "Alpaca", "Ferret",
#     "Chinchilla", "Great White Shark", "Blue Whale", "Killer Whale", "Sea Turtle", "Octopus", "Seahorse", "Clownfish", "Axolotl", "Manatee",
#     "Stingray", "Walrus", "Seal", "Narwhal", "Lobster", "Jellyfish", "Chameleon", "Komodo Dragon", "King Cobra", "Bearded Dragon",
#     "Iguana", "Tree Frog", "Monarch Butterfly", "Honey Bee", "Praying Mantis", "Ladybug", "Raccoon", "Camel", "Squirrel", "Buffalo"
# ]

animals = ["Earthworm"]

animal_urls = []

scraper = cloudscraper.create_scraper() 

print(f"{'ANIMAL':<25} | {'ESTADO':<10} | {'URL'}")
print("-" * 70)

for animal in animals:
    # 1. Limpieza del nombre para la URL:
    # Convertimos a minúsculas, quitamos paréntesis y cambiamos espacios por guiones
    clean_name = animal.lower().replace("(", "").replace(")", "").replace(" ", "-")
    url = f"https://a-z-animals.com/animals/{clean_name}/"

    try:
        # 2. Realizamos la petición
        # Usamos un pequeño delay para no ser agresivos con el servidor
        time.sleep(0.5) 
        response = scraper.get(url, timeout=10)

        # 3. Comprobación de existencia
        if response.status_code == 200:
            # Verificación adicional: ¿Está el nombre del animal en el título de la página?
            if clean_name.replace("-", " ") in response.text.lower():
                status = "EXISTE ✅"
            else:
                status = "DUDOSO ⚠️" # La página carga pero el contenido no parece ser el animal
        elif response.status_code == 404:
            status = "NO HALLADO ❌"
        else:
            status = f"ERROR {response.status_code}"

        print(f"{animal:<25} | {status:<10} | {url}")
        animal_urls.append((animal, url))
    except Exception as e:
        print(f"{animal:<25} | ERROR CRÍTICO: {str(e)}")

ANIMAL                    | ESTADO     | URL
----------------------------------------------------------------------
Earthworm                 | EXISTE ✅   | https://a-z-animals.com/animals/earthworm/


In [2]:
from bs4 import BeautifulSoup

animal_url = animal_urls[0][1]

response = scraper.get(animal_url)

print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

# Comprobar si se ha cargado bien el html de la página
soup.title

200


<title>Frog Animal Facts - Anura - A-Z Animals</title>

In [4]:
animal_text_container = soup.select_one("div#article-body")

first_child = animal_text_container.next_element
siblings = first_child.find_next_siblings()

siblings[-2].get("class")


AttributeError: 'NoneType' object has no attribute 'next_element'

In [18]:
from pathlib import Path
from bs4 import BeautifulSoup, NavigableString, Tag

animal_name = "Tiger"
url = "https://a-z-animals.com/animals/tiger/"
response = scraper.get(url, timeout=20)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
article = soup.select_one("div.article-body, div#article-body")

if article is None:
    raise ValueError("No se encontro el contenedor 'div.article-body' en la pagina.")

def clean_text(text: str) -> str:
    return " ".join(text.split())

def is_stop_container(node: Tag) -> bool:
    """Detiene el scraping al llegar a: <div class='container my-4'>."""
    if not isinstance(node, Tag) or node.name != "div":
        return False
    classes = set(node.get("class", []))
    return "container" in classes and "my-4" in classes

def to_markdown(node: Tag) -> list[str]:
    """Convierte nodos HTML relevantes a Markdown preservando secciones."""
    out = []

    if not isinstance(node, Tag):
        return out

    if is_stop_container(node):
        return out

    # Saltar bloques no textuales o irrelevantes para el contenido principal
    if node.name in {"script", "style", "figure", "img", "noscript", "svg", "aside"}:
        return out

    if node.name == "h2":
        text = clean_text(node.get_text(" ", strip=True))
        if text:
            out.append(f"## {text}")
            out.append("")
        return out

    if node.name == "h3":
        text = clean_text(node.get_text(" ", strip=True))
        if text:
            out.append(f"### {text}")
            out.append("")
        return out

    if node.name == "h4":
        text = clean_text(node.get_text(" ", strip=True))
        if text:
            out.append(f"#### {text}")
            out.append("")
        return out

    if node.name == "p":
        # Si el parrafo contiene strong como contenido principal, tratarlo como h4
        full_text = clean_text(node.get_text(" ", strip=True))
        strong_text = clean_text(" ".join(s.get_text(" ", strip=True) for s in node.find_all("strong")))

        if strong_text:
            is_strong_leading = full_text == strong_text or full_text.startswith(strong_text)
            remainder = clean_text(full_text[len(strong_text):]) if is_strong_leading else full_text
            has_meaningful_remainder = any(ch.isalnum() for ch in remainder)

            if is_strong_leading and not has_meaningful_remainder:
                out.append(f"#### {strong_text}")
                out.append("")
                return out

        if full_text:
            out.append(full_text)
            out.append("")
        return out

    if node.name in {"ul", "ol"}:
        items = node.find_all("li", recursive=False)
        for i, li in enumerate(items, start=1):
            text = clean_text(li.get_text(" ", strip=True))
            if not text:
                continue
            if node.name == "ul":
                out.append(f"- {text}")
            else:
                out.append(f"{i}. {text}")
        if items:
            out.append("")
        return out

    if node.name == "blockquote":
        text = clean_text(node.get_text(" ", strip=True))
        if text:
            out.append(f"> {text}")
            out.append("")
        return out

    # Para contenedores genericos, procesar hijos directos
    for child in node.children:
        if isinstance(child, NavigableString):
            text = clean_text(str(child))
            if text:
                out.append(text)
                out.append("")
            continue
        out.extend(to_markdown(child))

    return out

markdown_lines = []
for child in article.children:
    if isinstance(child, Tag) and is_stop_container(child):
        break
    if isinstance(child, Tag):
        markdown_lines.extend(to_markdown(child))

# Limpieza de saltos excesivos
cleaned_lines = []
prev_blank = False
for line in markdown_lines:
    is_blank = (line.strip() == "")
    if is_blank and prev_blank:
        continue
    cleaned_lines.append(line)
    prev_blank = is_blank

body_markdown = "\n".join(cleaned_lines).strip()
article_markdown = f"# {animal_name}\n\n{body_markdown}" if body_markdown else f"# {animal_name}"
print(article_markdown)  # preview
print(f"Total caracteres extraidos: {len(article_markdown)}")

# Guardar en data/tiger.md (funciona aunque el cwd sea scraping/)
output_path = Path("../data/tiger.md") if Path.cwd().name == "scraping" else Path("data/tiger.md")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(article_markdown, encoding="utf-8")
print(f"Guardado en: {output_path.resolve()}")

# Tiger

#### “No two tigers share the same pattern of stripes.”

Tigers are animals that live in both warm and cold areas of Asia. They are carnivores that hunt for prey at night. These big cats are solitary, and territorial, and are one of the world’s apex predators. A Siberian tiger can weigh up to 675 pounds. Males are bigger than females.

#### 5 Incredible Tiger Facts

- Tigers are good swimmers and love the water .
- They are hunted for their skin , fur, and other body parts.
- They mark their territory with urine to keep other tigers out.
- Their teeth measure about 4 inches long.
- This creature’s long tail helps it to keep its balance .

## Scientific Name

The scientific name of the tiger is Panthera tigris . The word Panthera means leopard and Tigris is Latin for tiger. They are sometimes called big cats. They belong to the Felidae family and the Mammalia class.

Nine subspecies include Sumatran , Siberian , Bengal , South China, Malayan , Indo-Chinese , Bali, Javan, and th

In [2]:
import re
from pathlib import Path
from bs4 import BeautifulSoup, NavigableString, Tag

# Si no existe un scraper en memoria, lo creamos
if "scraper" not in globals():
    import cloudscraper
    scraper = cloudscraper.create_scraper()

def clean_text(text: str) -> str:
    return " ".join(text.split())

def is_stop_container(node: Tag) -> bool:
    """Detiene el scraping al llegar a: <div class='container my-4'>."""
    if not isinstance(node, Tag) or node.name != "div":
        return False
    classes = set(node.get("class", []))
    return "container" in classes and "my-4" in classes

def html_to_markdown_sectioned(article: Tag) -> str:
    """Convierte el HTML del article-body a Markdown organizado por secciones."""
    def to_markdown(node: Tag) -> list[str]:
        out = []

        if not isinstance(node, Tag):
            return out

        if is_stop_container(node):
            return out

        if node.name in {"script", "style", "figure", "img", "noscript", "svg", "aside"}:
            return out

        if node.name == "h2":
            text = clean_text(node.get_text(" ", strip=True))
            if text:
                out.extend([f"## {text}", ""] )
            return out

        if node.name == "h3":
            text = clean_text(node.get_text(" ", strip=True))
            if text:
                out.extend([f"### {text}", ""] )
            return out

        if node.name == "h4":
            text = clean_text(node.get_text(" ", strip=True))
            if text:
                out.extend([f"#### {text}", ""] )
            return out

        if node.name == "p":
            # Si el parrafo contiene strong como contenido principal, tratarlo como h4
            full_text = clean_text(node.get_text(" ", strip=True))
            strong_text = clean_text(" ".join(s.get_text(" ", strip=True) for s in node.find_all("strong")))

            if strong_text:
                is_strong_leading = full_text == strong_text or full_text.startswith(strong_text)
                remainder = clean_text(full_text[len(strong_text):]) if is_strong_leading else full_text
                has_meaningful_remainder = any(ch.isalnum() for ch in remainder)

                if is_strong_leading and not has_meaningful_remainder:
                    out.extend([f"#### {strong_text}", ""] )
                    return out

            if full_text:
                out.extend([full_text, ""] )
            return out

        if node.name in {"ul", "ol"}:
            items = node.find_all("li", recursive=False)
            for i, li in enumerate(items, start=1):
                text = clean_text(li.get_text(" ", strip=True))
                if not text:
                    continue
                out.append(f"- {text}" if node.name == "ul" else f"{i}. {text}")
            if items:
                out.append("")
            return out

        if node.name == "blockquote":
            text = clean_text(node.get_text(" ", strip=True))
            if text:
                out.extend([f"> {text}", ""] )
            return out

        for child in node.children:
            if isinstance(child, NavigableString):
                text = clean_text(str(child))
                if text:
                    out.extend([text, ""] )
                continue
            out.extend(to_markdown(child))

        return out

    lines = []
    for child in article.children:
        if isinstance(child, Tag) and is_stop_container(child):
            break
        if isinstance(child, Tag):
            lines.extend(to_markdown(child))

    # Compactar lineas en blanco repetidas
    cleaned = []
    prev_blank = False
    for line in lines:
        is_blank = (line.strip() == "")
        if is_blank and prev_blank:
            continue
        cleaned.append(line)
        prev_blank = is_blank

    return "\n".join(cleaned).strip()

def slugify(name: str) -> str:
    slug = name.strip().lower().replace("-", " ")
    slug = re.sub(r"[^a-z0-9\s]", "", slug)
    slug = re.sub(r"\s+", "_", slug).strip("_")
    return slug or "animal"

def project_root_from_cwd() -> Path:
    cwd = Path.cwd()
    if (cwd / "data-animals").exists():
        return cwd
    if (cwd.parent / "data-animals").exists():
        return cwd.parent
    return cwd

def scrape_animal_article_markdown(url: str, animal_name: str) -> str:
    response = scraper.get(url, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    article = soup.select_one("div.article-body, div#article-body")
    if article is None:
        raise ValueError("No se encontro div.article-body")

    body_markdown = html_to_markdown_sectioned(article)
    return f"# {animal_name}\n\n{body_markdown}" if body_markdown else f"# {animal_name}"

# Puedes cambiar este limite para pruebas rapidas (ej. 5). None = todos.
MAX_ANIMALS = None

root = project_root_from_cwd()
output_dir = root / "data"
output_dir.mkdir(parents=True, exist_ok=True)

to_process = animal_urls[:MAX_ANIMALS] if MAX_ANIMALS else animal_urls
results = []
errors = []

print(f"Procesando {len(to_process)} animales...")

for idx, (animal_name, animal_url) in enumerate(to_process, start=1):
    try:
        md_text = scrape_animal_article_markdown(animal_url, animal_name)
        out_file = output_dir / f"{slugify(animal_name)}.md"
        out_file.write_text(md_text, encoding="utf-8")

        results.append((animal_name, str(out_file), len(md_text)))
        print(f"[{idx}/{len(to_process)}] OK  - {animal_name} -> {out_file.name} ({len(md_text)} chars)")
    except Exception as e:
        errors.append((animal_name, animal_url, str(e)))
        print(f"[{idx}/{len(to_process)}] ERR - {animal_name} -> {e}")

print("\nResumen")
print(f"Generados: {len(results)}")
print(f"Errores:   {len(errors)}")

if errors:
    print("\nPrimeros errores:")
    for animal_name, animal_url, err in errors[:10]:
        print(f"- {animal_name} | {animal_url} | {err}")

Procesando 1 animales...
[1/1] OK  - Earthworm -> earthworm.md (8986 chars)

Resumen
Generados: 1
Errores:   0


### Procesamiento de toda la información del tiburón blanco

1. Se acceder al div con id="single-animal-text" que es el que contiene todos los elementos con el texto de la información del animal.

2. Se obtienen todos los elementos dentro del div anterior

3. Se itera sobre cada elemento escribiendo en un fichero .md el texto correspondiente

In [ ]:
animal_text_container = soup.select_one("div#single-animal-text")

first_child = animal_text_container.next_element
siblings = first_child.find_next_siblings()

with open("./data/great_white_shark.md", 'w') as file:

    header = ""
    if first_child.name == "h2":
        header = "## "
        
    if first_child.name == "h3":
        header = "### "

    file.write(f"{header}{first_child.text}\n")

    for elem in siblings:
        if elem.name == "figure":
            continue
        
        if elem.get("class") is not None and elem.get("class")[0] == "fwpPitch":
            break
        
        header = ""
        if elem.name == "h2":
            header = "## "
            
        if elem.name == "h3":
            header = "### "

        if elem.name == "ul":
            items = elem.find_all("li")
            for i in items:
                file.write(f"   - {elem.text}\n")
            
            continue

        file.write(f"{header}{elem.text}\n")
